In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))

from src.data_loader import DataLoader
from src.metrics import Metrics
from src.utils import train_test_split_by_user
from sklearn.preprocessing import MultiLabelBinarizer
import pandas as pd

In [2]:
loader = DataLoader(path='../data')
ratings, movies, users = loader.load_all()

In [3]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy


In [4]:
movies['genres'] = movies['genres'].str.split('|')

mlb = MultiLabelBinarizer()
genres_matrix = mlb.fit_transform(movies['genres'])

genres_df = pd.DataFrame(
    genres_matrix,
    columns=mlb.classes_,
    index=movies['movieId']
)

genres_df.head()

,Action,Adventure,Animation,Children's,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
movieId,,,,,,,,,,,,,,,,,,
1,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0
3,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0
4,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity


similarity_df = pd.DataFrame(
    cosine_similarity(genres_df),
    index=genres_df.index,
    columns=genres_df.index
)

similarity_df.head()

movieId,1,2,3,4,5,6,7,8,9,10,...,3943,3944,3945,3946,3947,3948,3949,3950,3951,3952
movieId,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.333333,0.408248,0.408248,0.577350,0.0,0.408248,0.408248,0.0,0.000000,...,0.577350,0.408248,0.666667,0.000000,0.0,0.577350,0.000000,0.000000,0.000000,0.0
2,0.333333,1.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.816497,0.0,0.333333,...,0.000000,0.000000,0.666667,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0
3,0.408248,0.000000,1.000000,0.500000,0.707107,0.0,1.000000,0.000000,0.0,0.000000,...,0.707107,0.500000,0.000000,0.000000,0.0,0.707107,0.000000,0.000000,0.000000,0.0
4,0.408248,0.000000,0.500000,1.000000,0.707107,0.0,0.500000,0.000000,0.0,0.000000,...,0.707107,1.000000,0.000000,0.408248,0.0,0.707107,0.707107,0.707107,0.707107,0.5
5,0.577350,0.000000,0.707107,0.707107,1.000000,0.0,0.707107,0.000000,0.0,0.000000,...,1.000000,0.707107,0.000000,0.000000,0.0,1.000000,0.000000,0.000000,0.000000,0.0


In [19]:
def get_recommendations(user_id, train, similarity_df, k=10):

    user_data = train[train.userId == user_id]

    liked_movies = user_data[
        user_data.rating >= 4
    ]["movieId"].tolist()

    seen_movies = user_data["movieId"].tolist()

    scores = {}

    for movie in liked_movies:

        similar_movies = similarity_df[movie].nlargest(50)

        user_rating = user_data[
            user_data.movieId == movie
        ]["rating"].iloc[0]

        for sim_movie, score in similar_movies.items():

            if sim_movie in seen_movies:
                continue

            scores[sim_movie] = scores.get(sim_movie, 0) + score * user_rating

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    recommendations = [movie for movie, _ in ranked[:k]]

    return recommendations

In [10]:
train, test = train_test_split_by_user(ratings)
train.head()

,userId,movieId,rating,timestamp
0,1,1193,5,2000-12-31 22:12:40
1,1,661,3,2000-12-31 22:35:09
2,1,914,3,2000-12-31 22:32:48
5,1,1197,3,2000-12-31 22:37:48
6,1,1287,5,2000-12-31 22:33:59


In [25]:
metrics = Metrics()
total_prec = 0
total_rec = 0
total_ndcg = 0

uniq_users = 1000

for user in users['userId'].unique()[:uniq_users]:
    recommended = get_recommendations(user, train, similarity_df, 10)

    relevant = test[
        (test.userId == user) & (test.rating >= 4)
    ]['movieId'].tolist()

    if not len(relevant):
        continue

    # print("recommended:", recommended)
    # print("relevant:", relevant)
    # print("intersection:", set(recommended) & set(relevant))
    # print(train['movieId'].isin(similarity_df.index).all())

    precision, recall, ndcg = metrics.get_all(recommended, relevant, 10)
    
    total_prec += precision
    total_rec += recall
    total_ndcg += ndcg


print(f"total precision: {total_prec / uniq_users}\ntotal recall: {total_rec / uniq_users}\ntotal ndcg: {total_ndcg / uniq_users}")

total precision: 0.02080000000000001
total recall: 0.017128542066995035
total ndcg: 0.023011747268176168
